In [1]:
print("hello")

hello


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:

import pandas as pd
from pathlib import Path

# Questions based on the Attention Is All You Need paper
inputs = [
    "What is the main contribution of the paper 'Attention Is All You Need'?",
    "Why did the authors propose the Transformer instead of recurrent neural networks?",
    "How many layers are used in the Transformer encoder and decoder?",
    "What is scaled dot-product attention?",
    "Why does the Transformer use multi-head attention?",
    "Why are positional encodings needed in the Transformer?",
    "How many attention heads does the base Transformer use?",
    "Which optimizer and warmup schedule did the authors use?",
    "What BLEU score did the Transformer big model achieve on English-to-German translation?",
    "What is the main conclusion of the paper?"
]

outputs = [
    "The paper introduces the Transformer, a sequence transduction architecture based entirely on attention, without recurrent or convolutional layers.",
    "Recurrent networks process sequence positions sequentially, limiting parallelization. The Transformer uses attention to enable more parallel computation.",
    "Both the encoder and decoder contain six identical layers in the architecture described in the paper.",
    "Scaled dot-product attention calculates query-key dot products, scales them by the square root of the key dimension, applies softmax, and combines the values using the resulting weights.",
    "Multi-head attention lets the model attend to information from different representation subspaces and positions in parallel.",
    "Positional encodings provide token-order information because the Transformer does not use recurrence or convolution.",
    "The base Transformer uses eight attention heads.",
    "The authors used Adam with beta1=0.9, beta2=0.98, epsilon=1e-9, and 4,000 warmup steps.",
    "The Transformer big model achieved a BLEU score of 28.4 on the WMT 2014 English-to-German test task.",
    "The paper concludes that an attention-only Transformer can achieve strong machine-translation results while allowing more parallel training."
]

# Create dataset
qa_pairs = [
    {"question": q, "answer": a}
    for q, a in zip(inputs, outputs)
]

df = pd.DataFrame(qa_pairs)

# Save to the project's data directory, creating it if needed.
working_dir = Path.cwd().resolve()
project_root = next(
    (path for path in (working_dir, *working_dir.parents) if (path / "pyproject.toml").is_file()),
    working_dir,
)
data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)
csv_path = data_dir / "golden.csv"
df.to_csv(csv_path, index=False)

print(df.head())
print(f"Dataset saved to {csv_path.resolve()}")


                                            question  \
0  What is the main contribution of the paper 'At...   
1  Why did the authors propose the Transformer in...   
2  How many layers are used in the Transformer en...   
3              What is scaled dot-product attention?   
4  Why does the Transformer use multi-head attent...   

                                              answer  
0  The paper introduces the Transformer, a sequen...  
1  Recurrent networks process sequence positions ...  
2  Both the encoder and decoder contain six ident...  
3  Scaled dot-product attention calculates query-...  
4  Multi-head attention lets the model attend to ...  
Dataset saved to D:\Advance_Rag\data\golden.csv


In [5]:
from langsmith import Client

client = Client()
dataset_name = "AgenticAIReportGoldens"

# Store
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Input and expected output pairs for AgenticAIReport",
)
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)

{'example_ids': ['6142027a-2d5a-40a8-9495-71d20ad7d45e',
  '5ca0592a-188d-422d-ac1e-6c38e4c9adb4',
  '31b59c37-470d-4008-bee7-10afd40f50e2',
  'ec8e0285-3c59-4fa0-bc72-1b1ccabc619a',
  '7246598c-ff3d-4276-b246-a32825eb9d1c',
  '01ab3c7f-2c88-4834-bfe7-554c2c71fe2b',
  '05e52d09-58c6-4fde-b786-6deef8dabd79',
  'b399bd13-2a3f-4188-aedb-14c55b2e4728',
  '8d7a5eec-0307-42f5-9fe1-655304a9a91c',
  '217dbb84-1538-4719-9ebf-ed6adf9d9e2c'],
 'count': 10,
 'as_of': '2026-10-16T17:30:43.164195922Z'}

In [6]:
import sys
import os
from pathlib import Path

working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (working_dir, *working_dir.parents) if (path / "pyproject.toml").is_file()),
    working_dir,
)
sys.path.insert(0, str(PROJECT_ROOT))

from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor


# Simple file adapter for local file paths
class LocalFileAdapter:
    """Adapter for local file paths to work with ChatIngestor."""
    def __init__(self, file_path: str | Path):
        self.path = Path(file_path)
        self.name = self.path.name
    
    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str | Path = PROJECT_ROOT / "data" / "NIPS-2017-attention-is-all-you-need-Paper.pdf",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """
    Answer questions about the Agentic AI document using RAG.
    
    Args:
        inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
        data_path: Path to the Agentic AI text file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve
    
    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    """
    try:
        # Extract question from inputs
        question = inputs.get("question", "")
        if not question:
            return {"answer": "No question provided"}
        
        # Check if file exists
        if not Path(data_path).exists():
            return {"answer": f"Data file not found: {data_path}"}
        
        # Create file adapter
        file_adapter = LocalFileAdapter(data_path)
        
        # Build index using ChatIngestor
        ingestor = ChatIngestor(
            temp_base=str(PROJECT_ROOT / "data"),
            faiss_base=str(PROJECT_ROOT / "faiss_index"),
            use_session_dirs=True
        )
        
        # Build retriever
        ingestor.built_retriver(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k
        )
        
        # Get session ID and index path
        session_id = ingestor.session_id
        index_path = str(ingestor.faiss_dir.resolve())
        
        # Create RAG instance and load retriever
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index")
        )
        
        # Get answer
        answer = rag.invoke(question, chat_history=[])
        
        return {"answer": answer}
        
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}

In [7]:
# Test the function with a sample question
test_input = {"question": "What is the main contribution of the paper 'Attention Is All You Need'?"}
result = answer_ai_report_question(test_input)
print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])

{"timestamp": "2026-10-16T17:40:13.743301Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-10-16T17:40:13.744999Z", "level": "info", "event": "Loaded OPENAI_API_KEY from environment"}
{"keys": ["OPENAI_API_KEY"], "timestamp": "2026-10-16T17:40:13.745737Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-10-16T17:40:13.750542Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20261016_231013_a999497b", "temp_dir": "D:\\Advance_Rag\\data\\session_20261016_231013_a999497b", "faiss_dir": "D:\\Advance_Rag\\faiss_index\\session_20261016_231013_a999497b", "sessionized": true, "timestamp": "2026-10-16T17:40:13.752594Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "NIPS-2017-attention-is-all-you-need-Paper.pdf", "saved_as": "D:\\Advance_Rag\\data\\session_20261016_231013_a999497b\\b2dcf89b.pdf", "timestamp": "2026-10-16T17:40:13.757867Z", 

Question: What is the main contribution of the paper 'Attention Is All You Need'?

Answer: The main contribution of the paper 'Attention Is All You Need' is the introduction of the Transformer architecture, which is based solely on attention mechanisms and eliminates the need for recurrence and convolutions. This architecture demonstrates superior performance in machine translation tasks while being more parallelizable and requiring significantly less computational resources.


In [ ]:
from langsmith.evaluation import evaluate

In [9]:
from langchain_classic.evaluation import load_evaluator
from multi_doc_chat.utils.model_loader import ModelLoader

cot_qa_chain = load_evaluator("cot_qa", llm=ModelLoader().load_llm())

def cot_qa_evaluator(run, example):
    if example is None:
        raise ValueError("The cot_qa evaluator requires a dataset example.")

    result = cot_qa_chain.evaluate_strings(
        prediction=run.outputs["answer"],
        reference=example.outputs["answer"],
        input=example.inputs["question"],
    )
    return {
        "key": "cot_qa",
        "score": result["score"],
        "value": result["value"],
        "comment": result["reasoning"],
    }

dataset_name = "AgenticAIReportGoldens"

# Run evaluation using our RAG function
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=[cot_qa_evaluator],
    experiment_prefix="test-agenticAIReport-qa-rag",
    # Experiment metadata
    metadata={
        "variant": "RAG with FAISS and Attention Is All You Need",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

{"timestamp": "2026-10-16T17:59:18.802825Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-10-16T17:59:18.804171Z", "level": "info", "event": "Loaded OPENAI_API_KEY from environment"}
{"keys": ["OPENAI_API_KEY"], "timestamp": "2026-10-16T17:59:18.804665Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-10-16T17:59:18.808164Z", "level": "info", "event": "YAML config loaded"}
{"provider": "openai", "model": "gpt-4o-mini", "timestamp": "2026-10-16T17:59:18.809049Z", "level": "info", "event": "Loading LLM"}
{"timestamp": "2026-10-16T17:59:43.472876Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-10-16T17:59:43.473688Z", "level": "info", "event": "Loaded OPENAI_API_KEY from environment"}
{"keys": ["OPENAI_API_KEY"], "timestamp": "2026-10-16T17:59:43.474329Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "re

View the evaluation results for experiment: 'test-agenticAIReport-qa-rag-c41a91fc' at:
https://smith.langchain.com/o/13bd3280-fa80-41c8-83cb-f15df3335afc/datasets/2988b10c-dd03-40ea-b271-3de2ff95a4d4/compare?selectedSessions=5d31601a-71d0-418f-b185-b4a4dfa163aa




{"count": 11, "timestamp": "2026-10-16T17:59:43.812203Z", "level": "info", "event": "Documents loaded"}
{"chunks": 43, "chunk_size": 1000, "overlap": 200, "timestamp": "2026-10-16T17:59:43.814348Z", "level": "info", "event": "Documents split"}
{"model": "text-embedding-3-small", "timestamp": "2026-10-16T17:59:43.815496Z", "level": "info", "event": "Loading embedding model"}
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
{"added": 1, "index": "D:\\Advance_Rag\\faiss_index\\session_20261016_232943_d50ed68b", "timestamp": "2026-10-16T17:59:46.992283Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2026-10-16T17:59:46.993564Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2026-10-16T17:59:46.996000Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-10-16T17:59:46.996718Z", "level